# XGBoost Training vs Test RMSE Visualization

This notebook locates the current tuned XGBoost pipeline results and generates a publication-ready bar chart comparing training RMSE and hold-out test RMSE.

In [2]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

from config import MODEL_PARAMS, RANDOM_SEED
from data_loader import load_and_prepare
from feature_engineering import build_features
from weather import fetch_weather

## Locate Current Tuned XGBoost Results

Inspect project outputs and the current tuned XGBoost pipeline to determine the authoritative training and hold-out test RMSE values.

In [4]:
# Paths used by the current project pipeline
ROOT = Path.cwd()
TUNING_PATH = ROOT / "outputs" / "tuning_results.json"
TEST_METRICS_PATH = ROOT / "outputs" / "test_metrics.csv"

print(f"Tuning params file: {TUNING_PATH}")
print(f"Test metrics file: {TEST_METRICS_PATH}\n")

with open(TUNING_PATH, "r") as f:
    tuned_params = json.load(f)

print("Current tuned XGBoost parameters:")
print(json.dumps(tuned_params.get("XGBoost", {}), indent=2))

if TEST_METRICS_PATH.exists():
    df_test = pd.read_csv(TEST_METRICS_PATH)
    print("\nCurrent hold-out test metrics: ")
    print(df_test.to_string(index=False))
else:
    print("\nTest metrics file not found.")

Tuning params file: /Users/sanjana/Smart_dairyvisionAI/outputs/tuning_results.json
Test metrics file: /Users/sanjana/Smart_dairyvisionAI/outputs/test_metrics.csv

Current tuned XGBoost parameters:
{
  "subsample": 0.8,
  "reg_lambda": 2.0,
  "reg_alpha": 0.5,
  "n_estimators": 300,
  "min_child_weight": 1,
  "max_depth": 3,
  "learning_rate": 0.01,
  "colsample_bytree": 1.0
}

Current hold-out test metrics: 
     MAE     RMSE       R2  MAPE_pct
2.034751 2.574591 0.188041  9.864073


## Extract Training and Hold-out Test RMSE

Load the current data pipeline, apply the tuned XGBoost model, and compute the training RMSE and the hold-out test RMSE on the same split used by evaluation.

In [6]:
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# Load the same pipeline data used by evaluate.py
from config import RANDOM_SEED, MODEL_PARAMS

print("Loading data and preparing features...")
df_clean = load_and_prepare()
base_temp, base_hum = fetch_weather()
df_full, X, y = build_features(df_clean, base_temp, base_hum)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

params = MODEL_PARAMS.get("XGBoost", {}).copy()
params.update(tuned_params.get("XGBoost", {}))

model = XGBRegressor(**params, verbosity=0)
model.fit(X_train, y_train)

train_preds = model.predict(X_train)
test_preds = model.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))

print(f"\nComputed training RMSE: {train_rmse:.6f}")
print(f"Computed hold-out test RMSE: {test_rmse:.6f}")

train_rmse, test_rmse

Loading data and preparing features...

Computed training RMSE: 2.398453
Computed hold-out test RMSE: 2.574591


(np.float64(2.3984528705444164), np.float64(2.5745910319097227))

## Plot Publication-Ready Bar Chart

Create a single vertical bar chart with two bars for `Training` and `Test`, white background, exact RMSE labels, and y-axis starting at zero.

In [ ]:
# Inspect the tuned XGBoost model for existing per-round eval history
import joblib
from config import MODEL_PATH

print('Loading tuned XGBoost model from:', MODEL_PATH)
model = joblib.load(MODEL_PATH)

history = None
if hasattr(model, 'evals_result_'):
    history = model.evals_result_
    print('Found model.evals_result_')
else:
    try:
        booster = model.get_booster()
        if hasattr(booster, 'evals_result'):
            history = booster.evals_result() if callable(booster.evals_result) else booster.evals_result
            print('Found booster.evals_result')
    except Exception as exc:
        print('Booster inspect error:', exc)

if not history:
    raise RuntimeError(
        'Tuned XGBoost per-round eval history is unavailable in the current model. '
        'Cannot generate the requested training/validation RMSE curve.'
    )

print('Eval history keys:', list(history.keys()))

def find_curve_keys(history_dict):
    train_key = None
    valid_key = None
    for key in history_dict:
        lower = key.lower()
        if 'train' in lower and train_key is None:
            train_key = key
        if ('valid' in lower or 'validation' in lower) and valid_key is None:
            valid_key = key
    if not train_key and len(history_dict) == 2:
        keys = list(history_dict.keys())
        train_key, valid_key = keys[0], keys[1]
    if not valid_key and train_key:
        for key in history_dict:
            if key != train_key:
                valid_key = key
                break
    return train_key, valid_key

train_key, valid_key = find_curve_keys(history)
if train_key is None or valid_key is None:
    raise RuntimeError(
        'Unable to identify both training and validation eval sets in XGBoost history. '
        f'Found history keys: {list(history.keys())}'
    )

print('Training history key:', train_key)
print('Validation history key:', valid_key)

train_metrics = history[train_key]
valid_metrics = history[valid_key]

if isinstance(train_metrics, dict):
    if 'rmse' in train_metrics:
        train_rmse = train_metrics['rmse']
    else:
        train_rmse = next((v for v in train_metrics.values() if isinstance(v, list)), None)
else:
    train_rmse = train_metrics

if isinstance(valid_metrics, dict):
    if 'rmse' in valid_metrics:
        valid_rmse = valid_metrics['rmse']
    else:
        valid_rmse = next((v for v in valid_metrics.values() if isinstance(v, list)), None)
else:
    valid_rmse = valid_metrics

if train_rmse is None or valid_rmse is None:
    raise RuntimeError('Unable to extract RMSE lists from eval history.')

n_rounds = min(len(train_rmse), len(valid_rmse))
rounds = list(range(1, n_rounds + 1))
train_rmse = train_rmse[:n_rounds]
valid_rmse = valid_rmse[:n_rounds]

fig, ax = plt.subplots(figsize=(6.2, 4.0), dpi=300)
ax.plot(rounds, train_rmse, label='Training RMSE', color='#4C78A8', linewidth=2.2, marker='o', markersize=4)
ax.plot(rounds, valid_rmse, label='Validation RMSE', color='#F58518', linewidth=2.2, marker='o', markersize=4)

ax.set_title('XGBoost Training vs Validation RMSE', fontsize=14, fontweight='bold', pad=10)
ax.set_xlabel('Boosting Round', fontsize=11)
ax.set_ylabel('RMSE (L/day)', fontsize=11)
ax.set_xlim(1, n_rounds)
ax.set_axisbelow(True)
ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.legend(frameon=False, fontsize=10, loc='best')
fig.tight_layout(pad=0.8)

output_path = ROOT / 'outputs' / 'xgboost_training_validation_rmse_curve.png'
os.makedirs(output_path.parent, exist_ok=True)
fig.savefig(output_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved training/validation RMSE curve to:', output_path)


Saved figure to: /Users/sanjana/Smart_dairyvisionAI/outputs/xgboost_training_vs_test_rmse.png


PosixPath('/Users/sanjana/Smart_dairyvisionAI/outputs/xgboost_training_vs_test_rmse.png')

In [8]:
# Inspect the saved tuned XGBoost model for per-round evaluation history.
import joblib
from config import MODEL_PATH

print('Best model path:', MODEL_PATH)
model = joblib.load(MODEL_PATH)
print('Model type:', type(model))
print('Has model.evals_result_:', hasattr(model, 'evals_result_'))
if hasattr(model, 'evals_result_'):
    print('model.evals_result_ keys:', model.evals_result_.keys())
    print(model.evals_result_)
else:
    print('No model.evals_result_ attribute')

try:
    booster = model.get_booster()
    print('Booster type:', type(booster))
    has_booster = hasattr(booster, 'evals_result')
    print('Has booster.evals_result:', has_booster)
    if has_booster:
        res = booster.evals_result()
        print('Booster evals_result keys:', res.keys())
        print(res)
except Exception as exc:
    print('Booster inspection error:', exc)


Best model path: /Users/sanjana/Smart_dairyvisionAI/models/best_milk_model.pkl
Model type: <class 'xgboost.sklearn.XGBRegressor'>
Has model.evals_result_: False
No model.evals_result_ attribute
Booster type: <class 'xgboost.core.Booster'>
Has booster.evals_result: False


In [ ]:
import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from config import MODEL_PATH, OUTPUT_DIR
from data_loader import load_and_prepare
from feature_engineering import build_features
from weather import fetch_weather


In [14]:
# Run the current tuned XGBoost training pipeline with eval history and save the requested curve.
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from config import MODEL_PATH, RANDOM_SEED, OUTPUT_DIR
from data_loader import load_and_prepare
from feature_engineering import build_features
from weather import fetch_weather

# Load data and split using the current pipeline setup.
df_clean = load_and_prepare()
base_temp, base_hum = fetch_weather()
df_full, X, y = build_features(df_clean, base_temp, base_hum)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

# Load tuned model params from saved model and retrain with eval history.
model = joblib.load(MODEL_PATH)
params = model.get_params(deep=False)
params.pop('verbosity', None)
params.pop('callbacks', None)
params['eval_metric'] = 'rmse'

xgb = XGBRegressor(**params)
xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False,
)

history = xgb.evals_result_
print('Eval history keys:', list(history.keys()))
print('Training rounds:', len(history['validation_0']['rmse']))
print('Validation rounds:', len(history['validation_1']['rmse']))

train_rmse = history['validation_0']['rmse']
valid_rmse = history['validation_1']['rmse']
n_rounds = min(len(train_rmse), len(valid_rmse))
rounds = list(range(1, n_rounds + 1))

fig, ax = plt.subplots(figsize=(6.2, 4.0), dpi=300)
ax.plot(rounds, train_rmse[:n_rounds], label='Training RMSE', color='#4C78A8', linewidth=1.8)
ax.plot(rounds, valid_rmse[:n_rounds], label='Validation RMSE', color='#F58518', linewidth=1.8)
ax.set_title('XGBoost Training vs Validation RMSE', fontsize=14, fontweight='bold', pad=10)
ax.set_xlabel('Boosting Round', fontsize=11)
ax.set_ylabel('RMSE (L/day)', fontsize=11)
ax.set_xlim(1, n_rounds)
ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.legend(frameon=False, fontsize=10, loc='best')
fig.tight_layout(pad=0.8)
out_path = Path(OUTPUT_DIR) / 'xgboost_training_validation_rmse_curve.png'
fig.savefig(out_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved:', out_path)


Eval history keys: ['validation_0', 'validation_1']
Training rounds: 300
Validation rounds: 300
Saved: /Users/sanjana/Smart_dairyvisionAI/outputs/xgboost_training_validation_rmse_curve.png


In [12]:
from inspect import signature
from xgboost import XGBRegressor

print('XGBRegressor.fit signature:')
print(signature(XGBRegressor.fit))

model = joblib.load(MODEL_PATH)
params = model.get_params(deep=False)
print('Loaded model params keys:', list(params.keys()))
print('verbosity present:', 'verbosity' in params)
print('eval_metric supported by signature:', 'eval_metric' in signature(XGBRegressor.fit).parameters)
print('eval_set supported by signature:', 'eval_set' in signature(XGBRegressor.fit).parameters)


XGBRegressor.fit signature:
(self, X: Any, y: Any, *, sample_weight: Any | None = None, base_margin: Any | None = None, eval_set: Sequence[Tuple[Any, Any]] | None = None, verbose: bool | int | None = True, xgb_model: xgboost.core.Booster | str | ForwardRef('XGBModel') | None = None, sample_weight_eval_set: Sequence[Any] | None = None, base_margin_eval_set: Sequence[Any] | None = None, feature_weights: Any | None = None) -> 'XGBModel'
Loaded model params keys: ['objective', 'base_score', 'booster', 'callbacks', 'colsample_bylevel', 'colsample_bynode', 'colsample_bytree', 'device', 'early_stopping_rounds', 'enable_categorical', 'eval_metric', 'feature_types', 'feature_weights', 'gamma', 'grow_policy', 'importance_type', 'interaction_constraints', 'learning_rate', 'max_bin', 'max_cat_threshold', 'max_cat_to_onehot', 'max_delta_step', 'max_depth', 'max_leaves', 'min_child_weight', 'missing', 'monotone_constraints', 'multi_strategy', 'n_estimators', 'n_jobs', 'num_parallel_tree', 'random_st